# CourtListener search → DataFrame (HCDE 530)

Uses the **Week 4** folder and reads **`COURTLISTENER_API_TOKEN`** from **`week 4/.env`** (same variable as `week4_public_api.py`). Supports `KEY=value` or `export KEY=value` in `.env`.

Install packages into the **same** interpreter as this notebook (global kernel): `python3 -m pip install -r "week 4/requirements.txt"` — no virtualenv required.

Calls CourtListener **Legal Search API v4**: `GET /api/rest/v4/search/` with `type=o` (opinion clusters).

The API returns **`caseName`**, **`judge`**, and **`dateFiled`**. Plaintiff and defendant are **not** separate API fields; we **parse** them from `caseName` when it looks like *Party A v. Party B* (otherwise those columns are missing).

A second table (**Judgments classified by Judge names and jurisdiction**) pulls **20** opinions from the Seattle-area federal court (`court_id:wawd`) with **`jurisdiction`** set to **`Seattle jurisdiction`** on every row.

A third section (**Most recent judgments**) loads King County–related decisions from the **Washington Court of Appeals** (`court_id:washctapp` + phrase `"King County"`), sorts by decision date, and shows the **20 newest** rows (see notes there on coverage).


#

In [4]:
from __future__ import annotations

import json
import os
import urllib.error
import urllib.parse
import urllib.request
from pathlib import Path

import pandas as pd

# Find folder that contains `week 4/.env` or `.env` (kernel cwd varies)
HERE = Path.cwd()
for candidate in (HERE, HERE / "week 4", HERE.parent / "week 4"):
    if (candidate / ".env").is_file():
        HERE = candidate
        break
ENV_PATH = HERE / ".env"
TOKEN_ENV = "COURTLISTENER_API_TOKEN"
SEARCH_URL = "https://www.courtlistener.com/api/rest/v4/search/"


def load_dotenv_file(path: Path) -> None:
    if not path.is_file():
        return
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, val = line.partition("=")
        key = key.strip()
        if key.lower().startswith("export "):
            key = key[7:].strip()
        val = val.strip().strip('"').strip("'")
        if key and key not in os.environ:
            os.environ[key] = val


def split_parties(case_name: str) -> tuple[str, str]:
    if not case_name or not isinstance(case_name, str):
        return "", ""
    for sep in (" v. ", " V. ", " vs. ", " VS. ", " v ", " V "):
        if sep in case_name:
            left, right = case_name.split(sep, 1)
            return left.strip(), right.strip()
    return "", ""


def courtlistener_search(query: str, *, opinion_type: str = "o", token: str | None) -> dict:
    params = {"q": query, "type": opinion_type}
    url = SEARCH_URL + "?" + urllib.parse.urlencode(params)
    headers = {"Accept": "application/json; indent=2"}
    if token:
        headers["Authorization"] = f"Token {token}"
    req = urllib.request.Request(url, headers=headers)
    try:
        with urllib.request.urlopen(req, timeout=60) as resp:
            return json.loads(resp.read().decode("utf-8"))
    except urllib.error.HTTPError as e:
        detail = e.read().decode("utf-8", errors="replace")[:800]
        raise RuntimeError(f"CourtListener HTTP {e.code}: {detail}") from e


load_dotenv_file(ENV_PATH)
token = os.environ.get(TOKEN_ENV)
if not token:
    raise RuntimeError(
        f"Missing {TOKEN_ENV}. Add to {ENV_PATH} (see .env.example). Do not commit .env."
    )

QUERY = "Miranda v. Arizona"
raw = courtlistener_search(QUERY, token=token)
results = raw.get("results") or []
print("total matches:", raw.get("count"))
print("rows on first page:", len(results))


total matches: 47040
rows on first page: 20


In [2]:
rows: list[dict[str, object]] = []
for item in results[:25]:
    case = (item.get("caseName") or item.get("caseNameFull") or "").strip()
    plaintiff, defendant = split_parties(case)
    judge = (item.get("judge") or "").strip()
    panel = item.get("panel_names") or []
    if not judge and panel:
        judge = "; ".join(str(p) for p in panel if p)
    date_dec = item.get("dateFiled") or item.get("dateArgued")

    rows.append(
        {
            "case": case,
            "plaintiff": plaintiff or pd.NA,
            "defendant": defendant or pd.NA,
            "judge": judge or pd.NA,
            "date_of_decision": date_dec or pd.NA,
        }
    )

df = pd.DataFrame(rows)
df


,case,plaintiff,defendant,judge,date_of_decision
0,Miranda v. Arizona,Miranda,Arizona,"Consideration, Took",1969-10-13
1,Miranda v. Arizona,Miranda,Arizona,NaN,1965-11-22
2,Miranda v. Arizona,Miranda,Arizona,"Warren, Clark, Stewart, White, Harlan",1966-06-13
3,Best v. Miranda,Best,Miranda,"Brown, Swann, Thompson",2012-03-15
4,State v. Miranda-Cabrera,State,Miranda-Cabrera,"Snow, Timmer, Ehrlich",2004-12-02
5,State v. Miranda,State,Miranda,"Charles, Feldman, Jones, McGREGOR, Stanley, Th...",2001-05-04
6,"Callan, Miranda, Azuelo... v. Pimber","Callan, Miranda, Azuelo...",Pimber,NaN,2006-03-22
7,State v. Miranda,State,Miranda,"Timmer, Toci, Gerber",2000-09-28
8,State of Arizona v. Richard J. Glassel,State of Arizona,Richard J. Glassel,"Bales, Berch, Pelander",2013-11-21
9,People v. Miranda-Guerrero,People,Miranda-Guerrero,NaN,2022-11-17


## Judgments classified by Judge names and jurisdiction

Federal **Seattle** area opinions use CourtListener’s **`court_id:wawd`** (U.S. District Court, **Western District of Washington**). Each row sets **`jurisdiction`** to the label **`Seattle jurisdiction`**; **`court`** is the API’s full court name.

## Most recent judgments (King County)

CourtListener does **not** give King County **Superior** Court its own `court_id`. To capture **King County** decisions that are published in the corpus, this query uses the **Court of Appeals of Washington** (`court_id:washctapp`) plus the phrase **`"King County"`** (appeals from King County Superior Court and other King County matters often appear here).

The API returns pages in relevance order, so the code **follows `next`**, collects results, **sorts by `dateFiled` descending**, and keeps the **top 20** most recent opinions.

In [ ]:
from datetime import date


def _opinion_date(item: dict) -> date:
    ds = item.get("dateFiled") or ""
    parts = ds.split("-")
    if len(parts) != 3:
        return date.min
    y, m, d = (int(parts[0]), int(parts[1]), int(parts[2]))
    return date(y, m, d)


def collect_search_pages(q: str, *, token: str, max_pages: int = 25) -> list[dict]:
    """GET /search with pagination via `next` URL."""
    headers = {"Accept": "application/json; indent=2", "Authorization": f"Token {token}"}
    url = SEARCH_URL + "?" + urllib.parse.urlencode({"q": q, "type": "o"})
    acc: list[dict] = []
    for _ in range(max_pages):
        req = urllib.request.Request(url, headers=headers)
        with urllib.request.urlopen(req, timeout=60) as resp:
            payload = json.loads(resp.read().decode("utf-8"))
        acc.extend(payload.get("results") or [])
        url = payload.get("next")
        if not url or len(acc) >= 1200:
            break
    return acc


QUERY_KING_COUNTY = '"King County" court_id:washctapp'
raw_kc_count = courtlistener_search(QUERY_KING_COUNTY, token=token)
print("King County–indexed matches (Wash. Ct. App.):", raw_kc_count.get("count"))

king_items = collect_search_pages(QUERY_KING_COUNTY, token=token)
king_items.sort(key=_opinion_date, reverse=True)
king_top20 = king_items[:20]
print("rows paginated for sorting:", len(king_items), "· using newest", len(king_top20), "by dateFiled")

rows_recent: list[dict[str, object]] = []
for item in king_top20:
    case = (item.get("caseName") or item.get("caseNameFull") or "").strip()
    judge = (item.get("judge") or "").strip()
    panel = item.get("panel_names") or []
    if not judge and panel:
        judge = "; ".join(str(p) for p in panel if p)
    rows_recent.append(
        {
            "case": case or pd.NA,
            "judge": judge or pd.NA,
            "court": (item.get("court") or "").strip() or pd.NA,
            "court_id": (item.get("court_id") or "").strip() or pd.NA,
            "date_filed": item.get("dateFiled") or pd.NA,
        }
    )

df_recent = pd.DataFrame(rows_recent)
df_recent

In [3]:
SEATTLE_JURISDICTION_LABEL = "Seattle jurisdiction"

raw_seattle = courtlistener_search("court_id:wawd", token=token)
results_seattle = raw_seattle.get("results") or []
print("Seattle-area (W.D. Wash.) total matches:", raw_seattle.get("count"))
print("rows on this page (using first 20):", min(20, len(results_seattle)))

rows_jurisdiction: list[dict[str, object]] = []
for item in results_seattle[:20]:
    case = (item.get("caseName") or item.get("caseNameFull") or "").strip()
    judge = (item.get("judge") or "").strip()
    panel = item.get("panel_names") or []
    if not judge and panel:
        judge = "; ".join(str(p) for p in panel if p)
    court_name = (item.get("court") or "").strip()
    court_id = (item.get("court_id") or "").strip()
    date_dec = item.get("dateFiled") or item.get("dateArgued")

    rows_jurisdiction.append(
        {
            "judge": judge or pd.NA,
            "jurisdiction": SEATTLE_JURISDICTION_LABEL,
            "court": court_name or pd.NA,
            "court_id": court_id or pd.NA,
            "case": case or pd.NA,
            "date_of_decision": date_dec or pd.NA,
        }
    )

df_jurisdiction = pd.DataFrame(rows_jurisdiction)
df_jurisdiction

Seattle-area (W.D. Wash.) total matches: 3037
rows on this page (using first 20): 20


,judge,jurisdiction,court,court_id,case,date_of_decision
0,NaN,Seattle jurisdiction,"District Court, W.D. Washington",wawd,"Simmons v. Safeway, Inc.",2019-08-01
1,Settle,Seattle jurisdiction,"District Court, W.D. Washington",wawd,State v. Franciscan Health Sys.,2019-03-01
2,Leighton,Seattle jurisdiction,"District Court, W.D. Washington",wawd,"Animal Legal Def. Fund v. Olympic Game Farm, Inc.",2019-05-21
3,Jones,Seattle jurisdiction,"District Court, W.D. Washington",wawd,"Beane v. RPW Legal Servs., PLLC",2019-05-06
4,Lasnik,Seattle jurisdiction,"District Court, W.D. Washington",wawd,Galvez v. Cuccinelli,2019-07-17
5,Leighton,Seattle jurisdiction,"District Court, W.D. Washington",wawd,Mitchell v. Atkins,2019-05-20
6,Robart,Seattle jurisdiction,"District Court, W.D. Washington",wawd,Calderon-Rodriguez v. Wilcox,2019-02-06
7,Pechman,Seattle jurisdiction,"District Court, W.D. Washington",wawd,Gallupe v. Sedgwick Claims Mgmt. Servs. Inc.,2019-02-14
8,Lasnik,Seattle jurisdiction,"District Court, W.D. Washington",wawd,"United Statesi Ins. Servs. Nat'l, Inc. v. Ogden",2019-03-06
9,Leighton,Seattle jurisdiction,"District Court, W.D. Washington",wawd,Marine Carpenters Pension Fund v. Puglia Marin...,2019-04-10


### Notes

- **Pagination**: follow the `next` URL (cursor) for more pages; cache is ~10 minutes per CourtListener docs.
- **King County**: **Most recent judgments** uses `court_id:washctapp` + phrase `"King County"` because King County Superior Court is not its own CourtListener `court_id`; appellate opinions tied to King County supply recent dates.
- **Token**: same `COURTLISTENER_API_TOKEN` as `week4_public_api.py` in `week 4/.env`.
- **Docs**: [Legal Search API](https://www.courtlistener.com/help/api/rest/search/)
